# Installing required packages and Enviornment

In [ ]:
!pip install theoryminer
!pip install pandas

Checking the enviornment and installed packages!

Not important part, just checking of the version of packages!

In [ ]:
import os
import sys
import torch
import theoryminer
import random
import pandas as pd

random.seed(42)

package_dir = os.path.dirname(theoryminer.__file__)
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("theoryminer:", package_dir)
print("data files :", sorted(os.listdir(os.path.join(package_dir, "data"))))

assert os.path.isfile(os.path.join(package_dir, "data", "ELSST_R5.rdf.gz")), "the ELSST file is missing from the package"
print("check passed: the package holds the ELSST file")

Downlaoding sample papers

In [ ]:
import json
import urllib.request

REPO = "rasoulnorouzi/theoryminer"


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    return listing


listing = download_sample_files()
print(len(listing), "files:", sorted(item["name"] for item in listing))
assert len(listing) == 8, "expected 7 PDF files and SOURCES.md"
print("check passed: 8 files downloaded")

Not Important: This is just for showing how each function is defined!

In [ ]:
import importlib

H = importlib.import_module("theoryminer.harvest")

# Starting with theoryminer

## References Downloader!

In [ ]:
from theoryminer import extract_references

In [ ]:
extract_references??

In [ ]:
ref_dois = extract_references("sample_files/alrabai_2021_autonomy_supportive_teaching.pdf")
print(ref_dois)

In [ ]:
# now, a folder of papers and want to save as csv
ref_dois = extract_references("sample_files", save = "output/dois.csv")
print(ref_dois)

## Papaer Hunter: Downloading paper with doi

#### How it works

For each DOI: **1.** the PDF is already in the folder → skip.

#### The settings

| setting | what it does |
|---|---|
| `dois` | the output of `extract_references`, a list of DOIs, or a CSV file with a column `doi`. Duplicates are removed. |
| `output_dir` | the folder for the PDFs, made if missing. The file name is the DOI (`10.1093/pubmed/fdy133` → `10_1093_pubmed_fdy133.pdf`), so `harvest` later shows the DOI as `doc_id`. A PDF already there is not downloaded again: a stopped run can be run again. |
| `licences` | `None` (default): every open-access paper. `["cc-by"]`: only CC BY, whose text you may share. Other values: `cc-by-sa`, `cc-by-nc`, `cc-by-nd`, `cc-by-nc-sa`, `cc-by-nc-nd`, `public-domain`, `other-oa`, `publisher-specific-oa`, `none` (not open access), `unknown` (OpenAlex does not know the DOI). |
| `email` | your real email. OpenAlex answers faster with it, and Unpaywall does not work without it. `None` uses the variable `UNPAYWALL_EMAIL`. Pass it directly: a `.env` file is not always found. |
| `core_api_key` | a free key from [core.ac.uk](https://core.ac.uk): the author's own copy in a university repository, when the journal version is closed. `None` uses `CORE_API_KEY`; no key skips CORE. |
| `serpapi_key` | a paid SerpAPI key, for Google Scholar. `None` uses `SERPAPI_KEY`; no key skips it. |
| `save` | a CSV report: `"outputs/downloads.csv"`, or a folder (`"outputs"` → `outputs/downloads.csv`). `None` writes nothing. |

#### What you get back

One row per DOI, with `doi`, `status`, `source` (`openalex` or `tmsr-doi-downloader`), `licence`, `file` and
`note` (why it failed):

| status | meaning |
|---|---|
| `downloaded` | the PDF is in the folder |
| `already_there` | it was in the folder already |
| `skipped_licence` | its licence is not in your `licences` list |
| `not_found` | no open copy, or the site blocks robots |
| `error` | the network failed |

`not_found` is normal: many publishers block robots. A closed paper still comes through your library.


In [ ]:
from theoryminer import download_papers

In [ ]:
doi_list = [d['doi'] for d in ref_dois]

In [ ]:
ref_dois[:200]

In [ ]:
# download_papers("output/dois.csv", email="rsl@gmail.com", output_dir='doi_papers')
import random
ref_dois_samples = random.sample(ref_dois, 100)

download_papers(ref_dois_samples, email="rsl@gmail.com", output_dir='doi_papers')

## Harvesting Time!

#### How it works

`harvest` reads the text of each PDF page, cleans it in **ten steps**, and returns the clean sentences plus a
drop log. A step never deletes a line: it marks it as dropped, with a reason, so every removal can be checked.
The steps run on one paper at a time. A bad file in a folder never stops the run.

#### The ten steps

Steps 1 to 5 work on the **lines** of the page:

| # | step | the problem | example: before → after |
|---|---|---|---|
| 1 | `fix_unicode` | look-alike characters: `ﬁ` as one letter, curly quotes, long dashes, special hyphens, invisible marks | `The “effect” of self‑esteem — ﬁve items` → `The "effect" of self-esteem - five items` |
| 2 | `strip_markdown` | the page text carries bold, italic and link marks | `**Job insecurity** is linked to [poor health](https://…)` → `Job insecurity is linked to poor health` |
| 3 | `drop_running_heads` | the journal or author line printed on every page. Rule: a short line (up to 90 characters) that comes back 3 times or more | `Barrech et al. BMC Public Health (2018) 18:754` → dropped: `running_head` |
| 4 | `drop_junk_lines` | lines that are not running text. A heading also names the `section` of the lines below it. After a "References", "Notes" or "Acknowledgements" heading, all lines are dropped | `## Methods` → `heading` · `\| Mean \| SD \|` → `table_row` · `12` → `page_number` · `https://doi.org/…` → `doi_url` · reference list → `reference_section` |
| 5 | `join_paragraphs` | the PDF gives one row per printed line, and cuts words at the line end. A dropped line closes the paragraph; a paragraph without a full stop continues after a page break | `Job insecurity is a major stres-` + `sor for employees.` → `Job insecurity is a major stressor for employees.` |

Steps 6 to 10 work on **sentences**:

| # | step | the problem | example: before → after |
|---|---|---|---|
| 6 | `split_sentences` | the model reads one sentence at a time. The splitter knows academic abbreviations, so it does not cut at `e.g.`, `et al.`, `Fig.`, `pp.` | `Support, e.g. choice, raises motivation. Deci et al. showed this.` → 2 sentences |
| 7 | `strip_citations` | a citation names a source, not a construct. Brackets with a year and author names go; other brackets stay. Numbered citations like `[4, 5]` also stay (no year) | `Support buffers stress (Cohen & Wills, 1985).` → `Support buffers stress.` |
| 8 | `strip_stats` | test results are numbers, not constructs. Brackets that are mostly `=`, `<`, `p`, `n`, `CI`, `SD`, `%` go | `…poorer health (OR = 1.42, 95% CI 1.12-1.80, p < .01).` → `…poorer health.` |
| 9 | `drop_junk_sentences` | text that is not research text | licence text → `licence_text` · `Figure 2. …`, `Keywords: …` → `caption_or_keywords` · `R E S E A R C H` → `spaced_letters` · capitals only → `no_lowercase` · mostly digits → `digit_heavy` |
| 10 | `filter_length` | a very short sentence holds no claim; a very long one is often two sentences glued together, or a table | `See Table 2.` → `too_short` (under 4 words) · over 80 words → `too_long` |

Step 9 also puts a **flag** on doubtful sentences, but keeps them: `front_matter` (an editor name, a date or
an email mixed into the sentence), `fragment` (no full stop at the end), `long` (over 80 words).

On the 7 sample papers: 2,531 sentences kept, 1,865 rows dropped. The most common reasons: `table_row` (423),
`no_lowercase` (328), `reference_section` (247).

#### The settings

| setting | what it does |
|---|---|
| `source` | one PDF, or a folder of PDFs (not its subfolders) |
| `pages`, `skip_pages` | keep or skip pages: `"3-12"`, `[2, 5]` or `7`. Useful for one chapter of a book |
| `min_words`, `max_words` | the limits of step 10 (default 4 and 80) |
| `steps` | the steps, in order. Leave a step out, or add your own function |
| `cache` | keeps the page text in `<pdf>.pages.json` beside the PDF, so a second run is fast |

#### What you get back

- `sentences`: one row per sentence, with `doc_id` (the file name), `page`, `section`, `raw` (before steps 7
  and 8), `clean` (what the model reads) and `flag`.
- `dropped`: one row per dropped line or sentence, with its `drop_reason`. A file that is not a PDF, or a
  scanned PDF without text, gets one row: `not_pdf` or `unreadable_pdf`.


In [ ]:
from theoryminer.harvest import harvest

In [ ]:
harvest??

can we modify the steps? or settings?

of course! lets go with a toy example!

In [ ]:
theoryminer.DEFAULT_STEPS

In [ ]:
# defining what steps to be applied to the downloaded papers
workshop_steps = ['fix_unicode',
 'strip_markdown',
 'drop_running_heads',
 'drop_junk_lines',
 'join_paragraphs',
 'split_sentences',
 'strip_stats',
 'drop_junk_sentences',
 'filter_length']


# definig the steps for harvesting the sentences from the downloaded papers
my_harvest = harvest("doi_papers", steps=workshop_steps)

In [ ]:
# presenting the cleaned sentences!

my_harvest['sentences']

Lets back to the default steps!

In [ ]:
clean_sentences = harvest("doi_papers")

In [ ]:
clean_sentences['sentences']

# 2. CauseNet: finding cause and effect

`causenet()` reads each sentence. It finds the cause and the effect in the sentence.
It returns one row for each cause → effect pair. We call this row a **relation**.

The model is `rasoultilburg/SocioCausaNet`. It was trained on sentences from social-science papers.
It reads one sentence at a time. It does not read the sentence before or after.

Each relation keeps the path back to its source: the paper (`doc_id`), the page (`page`),
the sentence (`sent_id`, `sentence`) and its own number (`rel_id`).


## 2.1 The settings

| Setting | Default | What it does |
|---|---|---|
| `threshold` | `0.8` | The model gives each cause → effect pair a score from 0 to 1. A pair with a lower score is not kept. A lower value gives more relations and more errors. |
| `decision` | `"span_only"` | The rule that makes a sentence causal. `"span_only"`: the sentence has a cause span and an effect span. `"cls_only"`: only the classifier decides. |
| `mode` | `"neural"` | How the model links causes to effects. `"neural"`: the model scores every pair. `"auto"`: when a sentence has only one cause or only one effect, the model keeps every pair without a score. |
| `avoid_ambiguous` | `True` | Skip a relation when the cause or the effect only points to another sentence: "this", "it", "such things". The summary shows the skipped relations. |
| `device` | `"auto"` | Where the model runs. `"auto"` uses a GPU when there is one, else the CPU. You can also give `"cpu"`, `"cuda"` or `"mps"` (Apple). |
| `batch_size` | `64` | The number of sentences in one model call. Lower it after a GPU "out of memory" error. |

The first call loads the model. This takes a few seconds. The next calls use the model that is already in memory.

In [ ]:
from theoryminer import causenet
causenet??

Doing together a toy example!

In [ ]:

six_sentences = [
    "Social support increases feelings of security, satisfaction, self-worth, self-care, and well-being [52, 53].",
    "High levels of loneliness were strongly linked to poorer subjective health, lower life satisfaction, and increased psychosomatic complaints.",
    "Over time, discrimination can also contribute to allostatic load-a cumulative physiological burden imposed on the body through prolonged exposure to stress [7, 27].",
    "Furthermore, our results also support SDT's postulate that the relationship between perceived competence and the measured learning behaviors is mediated by intrinsic motivation.",
    "Vertical individualism was not significantly associated with anxiety at weekends although the association was positive.",
    "Secondly, social support can help to act as a buffer by reducing maladaptive emotional, physiological, and behavioral reactions to the event [52, 53, 55].",
]

six_relations = causenet(six_sentences, threshold=0.8, mode = 'neural', decision = 'span_only', batch_size = 8, avoid_ambiguous = True)

In [ ]:
six_relations

Back to the rail and working on our real samples!

In [ ]:
cause_effect_relations = causenet(clean_sentences['sentences'])

In [ ]:
random.sample(cause_effect_relations, 10)

# 3. Harmonizer: one construct, many names

Authors use different words for the same construct: `social support`, `strong social support`, `sources of social support`.
The harmonizer brings these names together in three steps:

| Step | Function | Needs a thesaurus |
|---|---|---|
| 3.1 Name each construct | `standardize_constructs()` | Yes (ELSST) |
| 3.2 Group the constructs | `pairwise_grouping()` or `clusterer()` | No |
| 3.3 Name each group | `label_groups()` or `standardize_groups()` | No / Yes |



## 3.1 Name each construct with ELSST

`standardize_constructs()` gives each span its closest ELSST concepts, best first.
The score is a similarity from 0 to 1. It is not a probability.

| Setting | Values | Default | What it does |
|---|---|---|---|
| `strategy` | `"leaf"`, `"path"`, `"anchor"`, `"context"`, `"bracket"`, `"enriched"` | `"enriched"` | The text that stands for each concept. See the table of strategies below. |
| `leaves_only` | `True`, `False` | `False` | `True` uses only the most specific concepts (2,661 entries). `False` uses all concepts (3,739 entries). |
| `top` | a whole number | `5` | The number of concepts for each span. |
| `threshold` | `None`, or a number from 0 to 1 | `None` | A minimum score. `None` keeps every match, also a weak one. |
| `embeddings` | `"allmpnet"`, `"bge_base"`, `"minilm"`, or a Hugging Face model name | `"allmpnet"` | The model that turns text into numbers. |
| `taxonomy` | `"elsst"`, or the path of a SKOS `.rdf` file | `"elsst"` | The thesaurus. |

**Warning:** each new combination of `strategy`, `leaves_only` and `embeddings` makes new thesaurus vectors the first time. This takes about 3 minutes.


### The six strategies

A strategy decides which text stands for a thesaurus concept. This is the text of the concept LONELINESS in each strategy:

| `strategy` | The text for LONELINESS |
|---|---|
| `"leaf"` | loneliness |
| `"path"` | well-being (society) > social support > loneliness |
| `"anchor"` | loneliness: well-being (society) > social support > loneliness |
| `"context"` | loneliness is related to social support, well-being (society) |
| `"bracket"` | loneliness (social support, well-being (society)) |
| `"enriched"` | loneliness. social isolation. well-being (society) > social support > loneliness |

`"enriched"` adds the other names of the concept ("social isolation") and its definition, when the thesaurus has one.

The best concept for 6 of the 15 spans, in each strategy:

| Span | `leaf` | `path` | `context` | `bracket` | `enriched` |
|---|---|---|---|---|---|
| high social support from classmates, … | SOCIAL SUPPORT 0.67 | SOCIAL SUCCESS 0.61 | SOCIAL SUPPORT 0.59 | SOCIAL SUPPORT 0.62 | SOCIAL SUCCESS 0.53 |
| teacher support | EDUCATIONAL SUPPORT PERSONNEL 0.72 | EDUCATIONAL SUPPORT PERSONNEL 0.65 | EDUCATIONAL SUPPORT PERSONNEL 0.67 | EDUCATIONAL SUPPORT PERSONNEL 0.73 | EDUCATIONAL SUPPORT PERSONNEL 0.59 |
| social trust | INTERPERSONAL TRUST 0.79 | TRUST 0.76 | TRUST 0.75 | INTERPERSONAL TRUST 0.77 | TRUST 0.76 |
| low levels of trust | INTERPERSONAL TRUST 0.68 | INTERPERSONAL TRUST 0.62 | INTERPERSONAL TRUST 0.63 | INTERPERSONAL TRUST 0.68 | TRUST 0.61 |
| depression | DEPRESSION 1.00 | DEPRESSION 0.62 | DEPRESSION 0.58 | DEPRESSION 0.71 | DEPRESSION 0.64 |
| symptoms of anxiety and depression | ANXIETY DISORDERS 0.64 | ANXIETY 0.60 | ANXIETY 0.56 | ANXIETY 0.61 | DEPRESSION 0.49 |

The scores of different strategies cannot be compared. `"leaf"` gives 1.00 when the span is the concept name.




## 3.2 Group the constructs (no thesaurus)

**`pairwise_grouping()`**

| Setting | Values | Default | What it does |
|---|---|---|---|
| `threshold` | a number from 0 to 1 | `0.69` | The similarity that a span needs to join a group. Higher = smaller groups. |
| `linkage` | `"average"`, `"complete"`, `"single"` | `"average"` | `"average"`: the mean similarity to the members. `"complete"`: the similarity to every member. `"single"`: the similarity to one member. |
| `embeddings` | `"allmpnet"`, `"bge_base"`, `"minilm"`, or a Hugging Face model name | `"allmpnet"` | The model that turns a span into numbers. The default `threshold` 0.69 was calibrated for `"allmpnet"`. |

**`clusterer()`**

| Setting | Values | Default | What it does |
|---|---|---|---|
| `min_cluster_size` | a whole number, 2 or more | `2` | The smallest group. |
| `min_samples` | a whole number, 1 or more | `2` | How dense a region must be. Higher = fewer groups and more noise. A noise span stays as its own group. |
| `umap` | `None`, or settings such as `dict(n_components=5)` | `None` | Reduce the vectors to fewer numbers before the clustering. |
| `embeddings` | as in `pairwise_grouping()` | `"allmpnet"` | |



## 3.3 Name each group

**`label_groups()`**: no thesaurus, no model

| Setting | Values | Default | What it does |
|---|---|---|---|
| `by` | `"keywords"`, `"medoid"`, `"both"` | `"keywords"` | `"keywords"`: the top words of the group. `"medoid"`: the most central span. `"both"`: the central span, then the keywords. |
| `top` | a whole number | `3` | The number of keywords. |

**`standardize_groups()`**: ELSST

| Setting | Values | Default | What it does |
|---|---|---|---|
| `name_by` | `"mean_sim"`, `"centroid"`, `"medoid"` | `"mean_sim"` | `"mean_sim"`: score each member, take the mean. `"centroid"`: score the mean of the members. `"medoid"`: score the central span only. |
| `strategy`, `leaves_only`, `top`, `threshold`, `embeddings`, `taxonomy` | as in 3.1 | as in 3.1 | |


In [ ]:
from theoryminer.harmonizer import standardize_constructs

standardize_constructs??

lets doing a toy example standardize_constructs and see how is working!

In [ ]:
cards = [
    "loneliness",
    "high levels of loneliness",
    "reducing loneliness",
    "alleviate feelings of loneliness",
    "social support",
    "strong social support",
    "high social support from classmates, teachers, and family",
    "teacher support",
    "social trust",
    "low levels of trust",
    "intrinsic motivation",
    "high intrinsic motivation",
    "low intrinsic motivation",
    "depression",
    "symptoms of anxiety and depression",
]

standardized_cards = standardize_constructs(cards, top = 3, strategy="context", leaves_only=True)
standardized_cards

In [ ]:
standardized_spans = standardize_constructs(cause_effect_relations, strategy="enriched", leaves_only=False, threshold=0.50)

In [ ]:
standardized_spans

Lets go for first grouping and then naming groups!

clustering!

In [ ]:
from theoryminer.harmonizer import clusterer

In [ ]:
clusterer??

In [ ]:
clustered_spans = clusterer(cause_effect_relations)

In [ ]:
for span, g in clustered_spans.items():
    print(f"{span} => (group {g['group']})")

Now the time is for grouping with pairwise_grouping!

In [ ]:
from theoryminer.harmonizer import pairwise_grouping

In [ ]:
pairwise_grouping??

In [ ]:
grouped_spans = pairwise_grouping(cause_effect_relations)

In [ ]:
for span, g in grouped_spans.items():
    print(f"{span} => (group {g['group']})")

Time for naming groups! both clustering and pairwise grouping methods!

Lets to do this with standardization!

In [ ]:
from theoryminer.harmonizer import standardize_groups
standardize_groups??

In [ ]:

sd_clustered = standardize_groups(clustered_spans, top = 3, strategy="enriched", leaves_only=False)

In [ ]:
sd_grouped = standardize_groups(grouped_spans, top = 3, strategy="enriched", leaves_only=False)

In [ ]:
sd_clustered

Now its time naming with keywords of the clusters/groups!

In [ ]:
from theoryminer.harmonizer import label_groups
label_groups??

In [ ]:
lb_clustered = label_groups(clustered_spans, top = 3)
lb_grouped = label_groups(grouped_spans, top = 3)

In [ ]:
lb_clustered

In [ ]:
multi_groups = {key: group for key, group in lb_clustered.items() if group["size"] > 5}
multi_groups

In [ ]:
multi_groups = {key: group for key, group in lb_grouped.items() if group["size"] > 5}
multi_groups


# 4. The causal map: from claims to a theory

`causal_map()` puts the relations and the names together:

- A **node** is a construct: one name from section 3.
- An **edge** is "cause → effect". It counts the papers and the sentences that make the claim.

Each edge keeps its relations, so you can go back to each sentence and paper.

| Function | What it does |
|---|---|
| `causal_map(relations, names)` | makes the nodes and the edges |
| `draw_map(cmap)` | shows the map as an interactive page |
| `save_map(cmap, folder)` | writes the tables (CSV) and a file for Gephi (GraphML) |
| `extract_dois(folder)` | finds the DOI and the title of each paper, for the links on the page |


## 4.1 The settings

**`causal_map(relations, names, ...)`**

| Setting | Values | Default | What it does |
|---|---|---|---|
| `names` | `labelled`, `elsst_names` or `concept_matches` | (required) | The names from section 3. Each gives a different map. |
| `min_sentences` | a whole number, 1 or more | `2` | Show an edge only when at least this many different sentences claim it. |
| `min_papers` | a whole number, 1 or more | `1` | Show an edge only when at least this many papers claim it. |
| `min_share` | a number above 0.5, up to 1 | `0.67` | For a two-way pair (A → B and B → A): the share of the papers that one side needs to be the main direction. |

An edge that a filter hides is not deleted. It goes to the list `hidden`, with the reason.

**`draw_map(cmap, ...)`**

| Setting | Values | Default | What it does |
|---|---|---|---|
| `papers` | `None`, or the result of `extract_dois()` | `None` | Show the DOI or the title of each paper. `None` shows the file name. |
| `save` | `None`, or a file name such as `"my_map.html"` | `None` | Also save the page as one HTML file. It works offline. |
| `title` | a text | `"Causal map"` | The title at the top of the page. |
| `height` | a number of pixels | `720` | The height of the map in the notebook. |


## 4.2 The small example: a map of the six sentences

We use the six sentences of section 2.3. First we name their spans, then we make the map.


## 4.4 Draw the map

`extract_dois()` reads your papers again to find their DOIs and titles. The page then links each sentence to its paper.
Give it the same folder as in section 1.



### Explore the page

1. **Search** a construct, for example "loneliness". The map shows it and its neighbours.
2. **Move the sliders** `min papers` and `min sentences`. The page has all the edges, also the hidden ones.
3. **Click an edge.** The panel shows every sentence that claims it, with a link to the paper.
4. **Choose one paper** in the list `Paper`. The map shows only its claims.
5. **Save a picture** (PNG, JPG or SVG) of the whole map, the current view, or a selection.

Node colours: green = cause only, blue = cause and effect, orange = effect only.


**Your task:** choose one edge from the table. Find it on the page. Read its sentences. Does the claim hold?


cmap = causal_map(case, names)
papers = extract_dois("sample_files")
draw_map(cmap, papers=papers, title="My causal map", save="my_map.html")

In [ ]:
from theoryminer import causal_map, draw_map, extract_dois, save_map

papers = extract_dois("doi_papers")

In [ ]:
cmap = causal_map(cause_effect_relations, sd_clustered)
draw_map(cmap, papers=papers, title="sd clustered", save="sd_clustered.html")
save_map(cmap, "sd_clustered")

In [ ]:
cmap = causal_map(cause_effect_relations, sd_grouped)
draw_map(cmap, papers=papers, title="sd grouped", save="sd_grouped.html")
save_map(cmap, "sd_grouped")

In [ ]:
cmap = causal_map(cause_effect_relations, lb_clustered)
draw_map(cmap, papers=papers, title="lb clustered", save="lb_clustered.html")
save_map(cmap, "lb_clustered")

In [ ]:
cmap = causal_map(cause_effect_relations, lb_grouped)
draw_map(cmap, papers=papers, title="lb grouped", save="lb_grouped.html")
save_map(cmap, "lb_grouped")